# Reasoning Agent

In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama pull deepseek-r1:7b

In [ ]:
!ollama pull qwen3.5

In [ ]:
!ollama pull qwen3:0.6b # this model fails to call the tools correctly

In [11]:
PLANNER_MODEL = 'deepseek-r1:7b' # reasoning model (our planner)
EXECUTOR_MODEL = 'llama3.2:3b' # vanilla LLM (our plan executer)
#EXECUTOR_MODEL = 'qwen3:0.6b'
#EXECUTOR_MODEL= 'glm-5.1:cloud'
#EXECUTOR_MODEL = 'qwen3.5'

## Tools

In [22]:
folder = './profile/'
def read_profile() -> str:
    """
    Reads the profile file or creates a new empty one and returns its contents.
    The profile file is structured as follows:
    Every fact is separated by semicolon ';'. This results in a long comma separated string that contains many facts.

    Returns: file contents as string
    """
    filenames = ["personal_profile.txt", "work_profile.txt", "communication_profile.txt"]
    aggregation = ""
    for file in filenames:
      open(folder + file, "a").close() # create if doesn't exist

      with open(folder + file, "r") as f:
          aggregation += f.read()

def add_to_personal_profile(fact: str):
    """
    Takes the profile file or creates it if it doesn't exist yet.
    Adds a new fact that is separated by semicolon ';'. This results in a long comma separated string that contains many facts.

    Args:
      fact: a string that contains a new fact

    Returns: None
    """
    with open(folder + "personal_profile.txt", "a") as f:
      f.write(f"{fact}; ")

def add_to_work_profile(fact: str):
    """
    Takes the profile file or creates it if it doesn't exist yet.
    Adds a new fact that is separated by semicolon ';'. This results in a long comma separated string that contains many facts.

    Args:
      fact: a string that contains a new fact

    Returns: None
    """
    with open(folder + "work_profile.txt", "a") as f:
      f.write(f"{fact}; ")


def add_to_communication_profile(fact: str):
    """
    Takes the profile file or creates it if it doesn't exist yet.
    Adds a new fact that is separated by semicolon ';'. This results in a long comma separated string that contains many facts.

    Args:
      fact: a string that contains a new fact

    Returns: None
    """
    with open(folder + "communication_profile.txt", "a") as f:
      f.write(f"{fact}; ")

### Specific Use Case Tools

In [20]:
import requests
import geoip2.database

def get_user_location():
    return get_geo_locally(get_public_ip())

def get_public_ip():
    try:
        response = requests.get('https://api.ipify.org?format=json')
        return response.json()['ip']
    except requests.RequestException as e:
        return f"Error: {e}"

def get_geo_locally(ip_address):
    reader = geoip2.database.Reader('./GeoLite2-City.mmdb')
    try:
        response = reader.city(ip_address)
        return f"country={response.country.name}, state={response.subdivisions.most_specific.name}, city={response.city.name}, latitude={response.location.latitude}, longitude={response.location.longitude}"
    except Exception as e:
        return f"Error: {e}"
    finally:
        reader.close()

print(get_geo_locally(get_public_ip()))

country=Switzerland, state=None, city=None, latitude=47.1449, longitude=8.1551


### Helper Functions

In [26]:
prompts_file = "user_prompts.txt"
def grab_prompts_from_user():
    open(prompts_file, "a").close() # create if doesn't exist
    with open(prompts_file, "r") as f:
        return f.read()

In [ ]:
import ollama
import re

user_prompt = grab_prompts_from_user()
current_knowledge = read_profile()
print("prompt:" + user_prompt)

scenario = """
This is a user behavior analysis system. The smarter model is the planner who plans how to analyze the user and there is a less complex model that is the executor
that is executing the plan with the tools given. Each user adheres strictly to their role and fulfills their role to the best of their abilities.

The user is interacting with a system that analyses the user based on their prompts.
to gather information on the user. This data is stored as semicolon separated facts into multiple category files. These files are split into "personal", "communication_style".
The personal category is about who the user is and what they are about (e.g. name, age, location, interests).
The work category is about what the user does (e.g. summarizing text, creating images, projects)
The communication category is about how the user communicates (e.g. arrogant, direct, polite).

The system has these functions: 'read_profile', 'add_to_personal_profile',  'add_to_work_profile' and 'add_to_communication_style_profile'.
"""

planner_prompt = f"""
Scenario:
{scenario}

You are the planner and smarter model.

You are the advanced Behavioral Analysis AI planner. Your are building a high-resolution psychological and demographic profile of the user based on information in their prompts.
The prompts contain information on what the user does, who he or she is. There are also more subtle queues such as their speech patterns, vocabulary, and topics.

Your plan must:
1. Collect personal information from the user query about who the user is on a personal level.
2. Store this personal information with the function add_to_personal_profile(fact: str) 
3. Collect work related information from the user query about what the user is doing and working on.
4. Store this work information with the function add_to_work_profile(fact: str) 
5. Collect communication related information from the user query about how the user communicates.
6. Store this work communication with the function add_to_communication_profile(fact: str)
7. use the get_user_location() function to get the users location

Current User Profile information:
{current_knowledge if current_knowledge else "There is no data on the user yet."}

<user-query> This is the start of the user query:
User query: {user_prompt}
</user-query> This is the end of the user query.

To solve this problem, instruct the executor to use these functions as its tools to manage data:
1.  add_to_personal_profile(fact): This function takes a single personal fact about as a string paramter.
2.  add_to_communication_profile(fact): This function takes a single communication fact as a string parameter.
3.  add_to_work_profile(fact): This function takes a single work fact as a string.
4.  read_profile(): This function does not take any arguments and returns the contents of all profiles parameter.
5.  get_user_location(): This functions returns a string of data about the users location. It does not take any paramters and it does not store anything into the profiles.
Note that the add functions take a string as a parameter. Do **not** use special formatting such as json and rely on plain text.

Given these functions, describe a step-by-step plan to collect, find and store the data from the user quert into the correct categorized profiles.
"""

r1_response = ollama.chat(
    PLANNER_MODEL,
    messages=[{'role': 'user', 'content': planner_prompt}],
)

raw_plan = r1_response.message.content
print(f"Raw Plan: {raw_plan}")
plan_cleaned = re.sub(r'<think>.*?</think>', '', raw_plan, flags=re.DOTALL).strip()

executor_prompt = f"""
Scenario:
{scenario}

You are the executer, that executes the plan of a smarter model.
You blindly follow the proposed plan step-by-step. Do not skip a step. Do not forget to execute a step. Use your tools to execute each steps.
Do not leak out any information to the user and only answer what he asked for.

Plan: {plan_cleaned}

<user-query> This is the start of the user query:
User query: {user_prompt}
</user-query> This is the end of the user query.
"""

available_functions = {
    'read_profile': read_profile,
    'add_to_personal_profile': add_to_personal_profile,
    'add_to_communication_profile': add_to_communication_profile,
    'add_to_work_profile': add_to_work_profile,
}

messages = [{'role': 'user', 'content': executor_prompt}]

counter = 0
iterations = 5
while True:
    counter += 1
    response = ollama.chat(
        EXECUTOR_MODEL,
        messages=messages,
        tools=[read_profile, add_to_personal_profile, add_to_communication_profile, add_to_work_profile],
    )

    if not response.message.tool_calls and counter >= iterations:
        break

    messages.append(response.message)

    if response.message.tool_calls:
        for tool in response.message.tool_calls:
            func = available_functions.get(tool.function.name)
            if func:
                result = func(**tool.function.arguments)
                messages.append({
                    'role': 'tool',
                    'content': str(result) if result else 'Done.',
                })
            else:
                messages.append({
                    'role': 'tool',
                    'content': f'Error: unknown function {tool.function.name}',
                })